## 1. Basic Tasks

**1. Create a table, make 3 changes to it (insert, update, insert), and use DESCRIBE HISTORY to review the
resulting versions.**

In [0]:
sales_df = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales.csv",
    header = True,
    inferSchema = True
)
sales_df.write.mode("overwrite").saveAsTable("dev.bronze.sales_day_7")

In [0]:
%sql
INSERT INTO dev.bronze.sales_day_7 VALUES
(79, 56, 1030, 5, 8, 50.0, 450.0, '2022-01-07'),
(80, 57, 1031, 1, 5, 40.0, 500.0, '2022-02-04')

In [0]:
%sql
UPDATE dev.bronze.sales_day_7 
SET quantity = 24
WHERE order_id = 10

In [0]:
%sql
INSERT INTO dev.bronze.sales_day_7 VALUES
(67, 49, 1032, 3, 7, 45.0, 470.0, '2022-03-17'),
(86, 57, 1033, 4, 6, 37.0, 430.0, '2022-05-23')

In [0]:
%sql
DESC HISTORY dev.bronze.sales_day_7

**2. Use COPY INTO to incrementally load 2 batches of files into a bronze table, confirming COPY INTO
doesn't reprocess the first batch.**

In [0]:
df = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales/sales_csv/sales.csv",
    header = True,
    inferSchema = True
)
df.display()

In [0]:
%sql
CREATE OR REPLACE TABLE dev.bronze.sales_copyinto_table (
    sale_id STRING,
    customer_id STRING,
    product_id STRING,
    quantity STRING,
    sale_amount STRING,
    sale_date STRING,
    region STRING
)

In [0]:
%sql
COPY INTO dev.bronze.sales_copyinto_table
FROM "/Volumes/dev/bronze/raw/sales/sales_csv/"
FILEFORMAT = CSV
FORMAT_OPTIONS("header" = "true")

For the first time `COPY INTO` it inserted `5000` rows that was there in the first `sales.csv` file in this volumn directory `"/Volumes/dev/bronze/raw/sales/sales_csv/"`, then I have uploaded another file in that directory of sales `sales-2.csv` and ran the `COPY INTO`
query again and it inserted only 15 rows from the new sales-2.csv, so it confirms `COPY INTO` doesn't reprocess the first batch.

**3. Query an old version of the table with both VERSION AS OF and TIMESTAMP AS OF.**

In [0]:
%sql
SELECT * FROM dev.bronze.sales_day_7 VERSION AS OF 1

In [0]:
%sql
SELECT * FROM dev.bronze.sales_day_7 TIMESTAMP AS OF "2026-08-26T07:04:48.000+00:00"

## 2. Intermediate Tasks

**4. Evolve the table's schema two ways: append a new column using mergeSchema, then change an
existing column's type using overwriteSchema; document the difference in what each requires.**

In [0]:
df_sales = spark.read.table("dev.silver.sales_cleaned_job")

In [0]:
from pyspark.sql.functions import lit

df_sales_addColumn = df_sales.withColumn("status", lit("pending"))

df_sales_addColumn.write.mode("append") \
.option("mergeSchema", "true") \
.saveAsTable("dev.silver.sales_cleaned_job")

In [0]:
from pyspark.sql.functions import col

df_sales_datatype = df_sales_addColumn.withColumn("order_id", col("order_id").cast("string")) \
    .withColumn("customer_id", col("customer_id").cast("string")) \
    .withColumn("transaction_id", col("transaction_id").cast("string")) \
    .withColumn("product_id", col("product_id").cast("string"))

df_sales_datatype.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dev.silver.sales_cleaned_job")

Difference between `mergeSchema` and `overwriteSchema`:
`mergeSchema` let's you append data in the current table, so if you add new column and then do `mergeSchema` the new data gets appended `mode("append")` with the extra column and for the previous data that was there in the table for the new column the value will be null. 
But if you change the schema of the current table and try to insert do `overwriteSchema` then you need to do `mode("overwrite")`, because you have altered the schema and the current data dose not match with the previous table schema. So the previous data gets lost.

**5. Set up an Autoloader stream ingesting from a folder, then drop 2 more files into the folder and
confirm they're picked up automatically.**

I have created the autoloader pipeline and the code for that I, will give below and after creating the autoloader pipeline, I have added that to the job and then added the file arrival trigger, so when ever I am adding any file to the current folder the pipeline runs and added the new data in the table. I am also going the provide the job.yml

job.yml
```
resources:
  jobs:
    job_autoloader_pipeline:
      name: job_autoloader_pipeline
      trigger:
        pause_status: UNPAUSED
        file_arrival:
          url: /Volumes/dev/bronze/raw/sales/sales_csv/
      tasks:
        - task_key: autoloader_pipeline
          pipeline_task:
            pipeline_id: 4599ac55-c16f-4ef9-b05b-ba1a9f375e93
      queue:
        enabled: true
      performance_target: PERFORMANCE_OPTIMIZED
```


In [0]:
# Pipeline code
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.table
def sales_cleaned_pipeline():
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .load("/Volumes/dev/bronze/raw/sales/sales_csv/")
    )

    df_cleaned = (
        df.drop("_rescued_data")
        .dropna(how="any")
        .dropDuplicates()
        .withColumn("sale_date", F.coalesce(F.col("sale_date"), F.current_date()))
    )

    return df_cleaned

**6. Use RESTORE to roll a table back to a version before a bad schema change, and describe what
happens to the versions that were created after the point you restored to.**

In [0]:
%sql
DESC HISTORY dev.bronze.sales_day_7

In [0]:
%sql
RESTORE TABLE dev.bronze.sales_day_7 TO VERSION AS OF 2;

In [0]:
%sql
select * from dev.bronze.sales_day_7

In [0]:
%sql
RESTORE TABLE dev.bronze.sales_day_7 TO VERSION AS OF 3;

In [0]:
%sql
SELECT * FROM dev.bronze.sales_day_7

## 3. Advanced Tasks

**7. Compare all four ingestion patterns covered (batch CTAS, COPY INTO, Autoloader, Lakeflow
Declarative Pipelines) on cost, latency, and operational complexity, and recommend which one
Cyntexa should use for a file source that arrives unpredictably throughout the day.**

Here is a comparison of the four ingestion patterns across cost, latency, and operational complexity, followed by the recommendation for Cyntexa's specific use case.

**Ingestion Pattern Comparison**

| Ingestion Pattern | Cost | Latency | Operational Complexity |
| --- | --- | --- | --- |
| **Batch CTAS / INSERT** | **Low:** Compute only runs when scheduled. However, costs can spike if poorly optimized queries rescan old files. | **High:** Data is only as fresh as the batch schedule (e.g., hourly, daily). | **High:** Requires manually building logic to identify new files (watermarking) to avoid reprocessing the entire dataset. |
| **COPY INTO** | **Low to Medium:** Highly efficient compute usage, but requires an external orchestrator (like Databricks Workflows) to trigger the job. | **Medium to High:** Bound by the orchestration schedule. Not real-time. | **Medium:** Automatically tracks processed files using the transaction log, but schema evolution requires manual configuration. |
| **Databricks Autoloader** | **Medium:** Running a 24/7 continuous stream costs more, but can be mitigated using `Trigger.AvailableNow` to process in micro-batches. | **Low:** Can process files in near real-time (seconds to minutes) as they land in cloud storage. | **Medium:** Requires managing stream checkpoints and writing PySpark Structured Streaming code, though it handles schema drift elegantly. |
| **Lakeflow (DLT)** | **Medium to High:** Premium compute costs for managed infrastructure, but automatically scales and optimizes resources. | **Low to Medium:** Supports both continuous execution for real-time latency and triggered execution. | **Low:** Highly declarative. Automatically manages checkpoints, task dependencies, data quality constraints, and infrastructure. |

**Recommendation for Cyntexa**

For a file source that arrives unpredictably throughout the day, **Databricks Autoloader** (ideally embedded within a **Lakeflow/Delta Live Tables** pipeline) is the definitive choice.

Here is why this fits Cyntexa's scenario:

* **Optimized File Detection:** Unlike traditional batch jobs that waste compute performing expensive directory listings to see if files *might* have arrived, Autoloader uses cloud-native file notification services (like AWS SNS/SQS or Azure Event Grid). It sits idle and only spins up processing power exactly when a file lands.
* **Cost vs. Latency Balance:** Because the files arrive unpredictably, scheduling a `COPY INTO` job every 15 minutes would result in paying for compute when no files exist. Autoloader solves this natively.
* **Schema Resilience:** Unpredictable file arrivals often come with unpredictable formatting changes. Autoloader's robust schema evolution and "rescue data" columns ensure the pipeline does not crash when upstream systems change.

**8. Design a recovery runbook: if a bad file corrupts the silver table at 2am, walk through the exact
commands (DESCRIBE HISTORY, RESTORE or time travel + overwrite) an on-call engineer would run.**

**1. Identify the Corruption Point**

The first step for the on-call engineer is to consult the Delta transaction log to pinpoint exactly when the bad file was ingested.

* Run the history command to view the table's timeline:
```sql
DESCRIBE HISTORY dev.silver.sales_clean;

```


* Review the `timestamp`, `operation`, and `operationParameters` columns to locate the write operation that occurred around 2:00 AM.
* Identify the `version` number *immediately preceding* the bad ingestion. For example, if Version 12 contains the 2:00 AM corruption, **Version 11** is your target healthy state.

**2. Verify the Clean State (Time Travel)**

Before committing to a rollback, the engineer should preview the target version to guarantee it is free of the corruption.

* Use Time Travel syntax to query the table exactly as it existed at Version 11:
```sql
SELECT * 
FROM dev.silver.sales_clean VERSION AS OF 11 
WHERE region IS NULL;

```


* If the version number is ambiguous, the engineer can also time travel using an exact timestamp just before the incident:
```sql
SELECT COUNT(*) 
FROM dev.silver.sales_clean TIMESTAMP AS OF '2026-08-26T01:50:00Z';

```



**3. Execute the Rollback (RESTORE)**

Once the healthy version is confirmed, the engineer will revert the table. Using the `RESTORE` command is the modern Delta Lake best practice, as it is highly optimized and automatically handles the underlying Parquet file pointers.

* Execute the restore command:
```sql
RESTORE TABLE dev.silver.sales_clean TO VERSION AS OF 11;

```


* *Alternative (Overwrite):* If the environment does not support the `RESTORE` command (e.g., an older Databricks runtime), the engineer can achieve the exact same result by overwriting the table with the time-traveled data:
```sql
INSERT OVERWRITE dev.silver.sales_clean
SELECT * FROM dev.silver.sales_clean VERSION AS OF 11;

```



**4. Isolate and Remediate**

Restoring the table mitigates the immediate outage, but the bad file still exists in the upstream Bronze layer.

* Pause the automated pipeline (e.g., Databricks Workflow) to prevent it from immediately re-ingesting the bad file on its next scheduled run.
* Locate the offending file in the Bronze directory and move it to a quarantine path.
* Resume the pipeline to process the remaining clean data.